In [1]:
# Cell 1
import os, joblib, time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Load train/test TF-IDF and labels saved earlier
X_train_tfidf, y_train = joblib.load(os.path.join(ARTIFACTS_DIR, "train_tfidf.joblib"))
X_test_tfidf,  y_test  = joblib.load(os.path.join(ARTIFACTS_DIR, "test_tfidf.joblib"))

print("Train shape:", X_train_tfidf.shape, "Test shape:", X_test_tfidf.shape)
print("Classes:", sorted(y_train.unique()))
classes = sorted(y_train.unique())
n_classes = len(classes)

Train shape: (11712, 8055) Test shape: (2928, 8055)
Classes: ['negative', 'neutral', 'positive']


In [4]:
# Cell 2: base model constructors (we will instantiate fresh models inside folds)
def make_nb():
    return MultinomialNB()

def make_lr():
    return LogisticRegression(
        C=2.0, max_iter=200, solver="lbfgs", multi_class="multinomial", n_jobs=-1
    )

def make_svm_calibrated():
    # LinearSVC inside CalibratedClassifierCV for probabilities
    base = LinearSVC(C=1.0, max_iter=2000)
    # 'sigmoid' or 'isotonic' calibration; sigmoid is safer for small data
    calibrated = CalibratedClassifierCV(estimator=base, cv=3, method='sigmoid')
    return calibrated

base_builders = [
    ("nb", make_nb),
    ("lr", make_lr),
    ("svm_cal", make_svm_calibrated)
]

In [5]:
# Cell 3: generate OOF probability features
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

n_train = X_train_tfidf.shape[0]
n_models = len(base_builders)

# We'll store probabilities for each model for each class: features shape = (n_train, n_models * n_classes)
oof_features = np.zeros((n_train, n_models * n_classes), dtype=np.float32)
oof_labels = np.array(y_train)

fold_idx = 0
start_time = time.time()
for train_idx, valid_idx in skf.split(np.zeros(n_train), y_train):
    fold_idx += 1
    print(f"\n--- Fold {fold_idx}/{k}  |  train {len(train_idx)}  valid {len(valid_idx)} ---")
    X_tr = X_train_tfidf[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train_tfidf[valid_idx]
    # For each base model, train on X_tr and predict_proba on X_val
    col_offset = 0
    for m_idx, (name, builder) in enumerate(base_builders):
        print(f" Training base model: {name} ...", end=" ")
        model = builder()
        # fit (CalibratedClassifierCV will internally calibrate)
        model.fit(X_tr, y_tr)
        # get probabilities on validation fold
        prob = model.predict_proba(X_val)  # shape (len(valid_idx), n_classes)
        # Ensure ordering of classes matches 'classes' variable
        # CalibratedClassifierCV and scikit-learn models return columns in model.classes_
        # We'll reorder columns to match global classes
        model_classes = list(model.classes_)
        if model_classes != classes:
            # reorder columns
            reorder_idx = [model_classes.index(c) for c in classes]
            prob = prob[:, reorder_idx]
        # place into oof_features
        oof_features[valid_idx, col_offset:col_offset + n_classes] = prob
        col_offset += n_classes
        print("done.")
end_time = time.time()
print(f"\nOOF generation done in {end_time - start_time:.1f}s")
print("OOF features shape:", oof_features.shape)


--- Fold 1/5  |  train 9369  valid 2343 ---
 Training base model: nb ... done.
 Training base model: lr ... 

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


done.
done.ning base model: svm_cal ... 

--- Fold 2/5  |  train 9369  valid 2343 ---
 Training base model: nb ... done.
 Training base model: lr ... 

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


done.
done.ning base model: svm_cal ... 

--- Fold 3/5  |  train 9370  valid 2342 ---
 Training base model: nb ... done.
 Training base model: lr ... 

...

done.
done.ning base model: svm_cal ... 

--- Fold 4/5  |  train 9370  valid 2342 ---
 Training base model: nb ... done.
 Training base model: lr ... 

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


done.
done.ning base model: svm_cal ... 

--- Fold 5/5  |  train 9370  valid 2342 ---
 Training base model: nb ... done.
 Training base model: lr ... 

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


done.
done.ning base model: svm_cal ... 

OOF generation done in 13.3s
OOF features shape: (11712, 9)


In [6]:
# Cell 4
joblib.dump((oof_features, oof_labels, classes), os.path.join(ARTIFACTS_DIR, "oof_train_features.joblib"))
print("Saved OOF features to artifacts/oof_train_features.joblib")
# Quick spot-check: rows should sum to n_models (sum of probs across classes per model = 1)
per_model_sums = oof_features.reshape(n_train, n_models, n_classes).sum(axis=2)  # shape (n_train, n_models)
print("Per-model probability sums (first 5 rows):\n", per_model_sums[:5])

Saved OOF features to artifacts/oof_train_features.joblib
Per-model probability sums (first 5 rows):
 [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


In [7]:
# Cell 5: train meta-model (Logistic Regression)
oof_X, oof_y, classes = joblib.load(os.path.join(ARTIFACTS_DIR, "oof_train_features.joblib"))
meta = LogisticRegression(C=1.0, max_iter=200, multi_class='multinomial', solver='lbfgs', n_jobs=-1)
meta.fit(oof_X, oof_y)
print("Trained meta-model (LogisticRegression).")
# save meta model
joblib.dump((meta, classes), os.path.join(ARTIFACTS_DIR, "meta_model.joblib"))
print("Saved meta_model.joblib")

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Trained meta-model (LogisticRegression).
Saved meta_model.joblib


In [8]:
# Cell 6: fit base models on full training set and save
full_models = {}
for name, builder in base_builders:
    print(f"Fitting full model: {name} ...", end=" ")
    m = builder()
    m.fit(X_train_tfidf, y_train)
    path = os.path.join(ARTIFACTS_DIR, f"{name}_full.joblib")
    joblib.dump((m, classes), path)
    full_models[name] = m
    print(f"saved -> {path}")
print("All base models trained on full training set and saved.")

Fitting full model: nb ... saved -> artifacts\nb_full.joblib
Fitting full model: lr ... 

C:\Users\Dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


saved -> artifacts\lr_full.joblib
saved -> artifacts\svm_cal_full.joblib
All base models trained on full training set and saved.


In [9]:
# Cell 7: build meta-features for test set (concat prob columns from each full base model)
test_meta_feats = np.zeros((X_test_tfidf.shape[0], n_models * n_classes), dtype=np.float32)
col_offset = 0
for m_idx, (name, _) in enumerate(base_builders):
    model, model_classes = joblib.load(os.path.join(ARTIFACTS_DIR, f"{name}_full.joblib"))
    prob = model.predict_proba(X_test_tfidf)
    # reorder columns to match global classes if needed
    model_classes = list(model.classes_)
    if model_classes != classes:
        reorder_idx = [model_classes.index(c) for c in classes]
        prob = prob[:, reorder_idx]
    test_meta_feats[:, col_offset:col_offset+n_classes] = prob
    col_offset += n_classes

# meta-model predict
meta, _ = joblib.load(os.path.join(ARTIFACTS_DIR, "meta_model.joblib"))
y_pred_stack = meta.predict(test_meta_feats)
y_pred_stack_proba = meta.predict_proba(test_meta_feats).max(axis=1)

# Also get best base model predictions for comparison (Logistic Regression)
lr_model, _ = joblib.load(os.path.join(ARTIFACTS_DIR, "lr_full.joblib"))
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluate
print("Stacking Ensemble Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print(classification_report(y_test, y_pred_stack))

print("\nBest base (Logistic Regression) Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Confusion matrix for stacking
print("\nConfusion matrix (stacking):")
print(confusion_matrix(y_test, y_pred_stack, labels=classes))

Stacking Ensemble Results:
Accuracy: 0.7844945355191257
              precision    recall  f1-score   support

    negative       0.82      0.92      0.87      1835
     neutral       0.62      0.51      0.56       620
    positive       0.79      0.62      0.70       473

    accuracy                           0.78      2928
   macro avg       0.74      0.68      0.71      2928
weighted avg       0.78      0.78      0.78      2928


Best base (Logistic Regression) Results:
Accuracy: 0.7756147540983607
              precision    recall  f1-score   support

    negative       0.80      0.93      0.86      1835
     neutral       0.62      0.47      0.54       620
    positive       0.80      0.56      0.66       473

    accuracy                           0.78      2928
   macro avg       0.74      0.66      0.69      2928
weighted avg       0.77      0.78      0.76      2928


Confusion matrix (stacking):
[[1686  111   38]
 [ 262  316   42]
 [  98   80  295]]


In [10]:
# Cell 8: confirm saved artifacts list
print("Artifacts saved in", ARTIFACTS_DIR)
print(sorted(os.listdir(ARTIFACTS_DIR)))

Artifacts saved in artifacts
['linear_svm_model.joblib', 'logistic_regression_model.joblib', 'lr_full.joblib', 'meta_model.joblib', 'naive_bayes_model.joblib', 'nb_full.joblib', 'oof_train_features.joblib', 'svm_cal_full.joblib', 'test_tfidf.joblib', 'tfidf_vectorizer.joblib', 'train_tfidf.joblib']
